Task: Automate Scraping 

Automatically collect military strength data for all countries from Global Firepower and store it in a structured dataset for analysis.

Stage-1:Access Countries listing pages

Requested the Global Firepower countries listing page to retrieve links for each country’s military profile.

In [ ]:
#Step-1: Read URL's from file
import requests
from bs4 import BeautifulSoup

# page containing all countries
url = "https://www.globalfirepower.com/countries-listing.php"

# request webpage
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

Stage-2:Extract Country links

Extracted country profile links from the webpage and built complete URLs for automated data collection.

In [ ]:
# extract country links
links = soup.select("a[href*='country-military-strength-detail.php']")

# build full URLs
urls = [
    "https://www.globalfirepower.com/" + link["href"]
    for link in links
]

Stage-3: Remove Duplicate url's

Removed duplicate links and sorted the list to ensure each country is scraped only once.

In [ ]:
# remove duplicates
urls = sorted(set(urls))

print("Total countries:", len(urls))
print(urls[:5])

Stage-4: Loop through country pages

Iterated through each country URL to systematically scrape military strength data.

In [ ]:
# Loop through URL's and scrape data
import re
import pandas as pd
import time

HEADERS = {"User-Agent": "Mozilla/5.0"}

all_data = []

for i, url in enumerate(urls):

    print(f"Scraping {i+1}/{len(urls)}")

    soup = BeautifulSoup(requests.get(url, headers=HEADERS).text, "html.parser")
    text = soup.get_text()

    # extract country name & remove year
    title = soup.find("title").text
    country_name = re.sub(r"^\d{4}\s*", "", title)
    country_name = country_name.split("Military Strength")[0].strip()

    data = {"country": country_name}

    # patterns for extraction
    patterns = {

        # Power Index
        "power_index": r"Power Index score of\s+([\d.]+)",

        # Population & Economy
        "total_population": r"Total Population:\s+([\d,]+)",
        "gdp": r"Purchasing Power Parity:\s+(\$[\d,]+)",
        "defense_budget": r"Defense Budget:\s+(\$[\d,]+)",
        "external_debt": r"External Debt:\s+(\$[\d,]+)",

        # Manpower
        "total_manpower": r"Available Manpower\s+([\d,]+)",
        "active_personnel": r"Active Personnel\s+([\d,]+)",
        "reserve_personnel": r"Reserve Personnel\s+([\d,]+)",

        # Personnel Distribution
        "air_force_personnel": r"Air Force\s+([\d,]+)",
        "army_personnel": r"Army\s+([\d,]+)",
        "navy_personnel": r"Navy\s+([\d,]+)",

        # Air Power
        "total_aircraft": r"Aircraft Total:\s+Stock:\s+([\d,]+)",
        "fighter_aircraft": r"Fighters:\s+Stock:\s+([\d,]+)",
        "attack_aircraft": r"Attack:\s+Stock:\s+([\d,]+)",
        "transport_aircraft": r"Transport:\s+Stock:\s+([\d,]+)",
        "trainer_aircraft": r"Trainers:\s+Stock:\s+([\d,]+)",
        "special_mission_aircraft": r"Special-Mission:\s+Stock:\s+([\d,]+)",
        "helicopters": r"Helicopters:\s+Stock:\s+([\d,]+)",
        "attack_helicopters": r"Attack Helicopters:\s+Stock:\s+([\d,]+)",
        "tanker_aircraft": r"Tankers:\s+Stock:\s+([\d,]+)",

        # Naval Power
        "naval_assets": r"Total Assets:\s+([\d,]+)",
        "aircraft_carriers": r"Aircraft Carriers:\s+([\d,]+)",
        "helicopter_carriers": r"Helicopter Carriers:\s+([\d,]+)",
        "destroyers": r"Destroyers:\s+([\d,]+)",
        "frigates": r"Frigates:\s+([\d,]+)",
        "corvettes": r"Corvettes:\s+([\d,]+)",
        "submarines": r"Submarines:\s+([\d,]+)",
        "patrol_vessel": r"Patrol Vessels:\s+([\d,]+)",
        "mine_warfare": r"Mine Warfare:\s+([\d,]+)",

        # Land Power (Stock & Readiness)
        "tanks": r"Tanks:\s+Stock:\s+([\d,]+)(?:\s+Readiness:\s+([\d,]+))?",
        "self_propelled_artillery": r"Self-Propelled Artillery:\s+Stock:\s+([\d,]+)(?:\s+Readiness:\s+([\d,]+))?",
        "towed_artillery": r"Towed Artillery:\s+Stock:\s+([\d,]+)(?:\s+Readiness:\s+([\d,]+))?",
        "rocket_artillery": r"Rocket Projectors:\s+Stock:\s+([\d,]+)(?:\s+Readiness:\s+([\d,]+))?"
    }

    # extract values
    for key, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            if match.lastindex and match.lastindex > 1:
                data[key + "_stock"] = match.group(1)
                data[key + "_readiness"] = match.group(2)
            else:
                data[key] = match.group(1)

    all_data.append(data)

    time.sleep(1)

# convert to DataFrame
df = pd.DataFrame(all_data)
df.to_csv("military_ready.csv", index=False)

Total countries: 145
['https://www.globalfirepower.com//country-military-strength-detail.php?country_id=afghanistan', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=albania', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=algeria', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=angola', 'https://www.globalfirepower.com//country-military-strength-detail.php?country_id=argentina']
Scraping 1/145
Scraping 2/145
Scraping 3/145
Scraping 4/145
Scraping 5/145
Scraping 6/145
Scraping 7/145
Scraping 8/145
Scraping 9/145
Scraping 10/145
Scraping 11/145
Scraping 12/145
Scraping 13/145
Scraping 14/145
Scraping 15/145
Scraping 16/145
Scraping 17/145
Scraping 18/145
Scraping 19/145
Scraping 20/145
Scraping 21/145
Scraping 22/145
Scraping 23/145
Scraping 24/145
Scraping 25/145
Scraping 26/145
Scraping 27/145
Scraping 28/145
Scraping 29/145
Scraping 30/145
Scraping 31/145
Scraping 32/145
Scraping 3

This script successfully:

Successfully automated the extraction of global military strength data and stored it in a structured format ready for cleaning and analysis.

Task: Data Cleaning & Standardization

Clean and standardize the raw military dataset by separating stock/readiness values, converting financial and numeric fields, handling missing data, and preparing a structured dataset ready for KPI analysis and dashboard visualization.
Stage 1: Load Raw Dataset

Loaded the raw military dataset(military_raw_data.csv) into a DataFrame to begin cleaning and structuring.

In [ ]:
import pandas as pd
import re

# load raw data
df = pd.read_csv("military_raw_data.csv")

Stage 2: Split Stock & Readiness Values

Combined values in tanks, self_propelled_artillery, towed_artillery, and rocket_artillery were split into separate stock and readiness columns for clearer strength analysis.

In [ ]:
# -----------------------------------
# FUNCTION TO SPLIT STOCK & READINESS
# -----------------------------------
def split_values(column):

    df[column + "_stock"] = df[column].str.extract(r"Stock:\s*(\d+)")
    df[column + "_readiness"] = df[column].str.extract(r"Readiness:\s*(\d+)")

    # convert to numeric
    df[column + "_stock"] = pd.to_numeric(df[column + "_stock"], errors="coerce")
    df[column + "_readiness"] = pd.to_numeric(df[column + "_readiness"], errors="coerce")

    # drop original messy column
    df.drop(column, axis=1, inplace=True)
Extraction of ready and stock values:
# columns containing stock & readiness
cols_to_split = [
    "tanks",
    "self_propelled_artillery",
    "towed_artillery",
    "rocket_artillery"
]

for col in cols_to_split:
    split_values(col)

Stage 3: Clean Money Columns

Removed symbols and commas from gdp, defense_budget, and external_debt, converting them into numeric values for accurate financial comparison.

In [ ]:
# -----------------------------------
# CLEAN MONEY COLUMNS
# -----------------------------------
money_cols = ["gdp", "defense_budget", "external_debt"]

for col in money_cols:
    df[col] = df[col].str.replace("$", "", regex=False)
    df[col] = df[col].str.replace("USD", "", regex=False)
    df[col] = df[col].str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

Stage 4: Remove Commas from Other Numeric Columns

Comma formatting was removed from numeric columns to ensure consistent numeric conversion.

In [ ]:
# -----------------------------------
# REMOVE COMMAS FROM OTHER NUMERIC COLUMNS
# -----------------------------------
for col in df.columns:
    if col != "country":
        df[col] = df[col].astype(str).str.replace(",", "", regex=False)

Stage 5: Convert Columns to Numeric

All columns except country were converted to numeric types to enable calculations and KPI analysis.

In [ ]:
# convert numeric columns
for col in df.columns:
    if col != "country":
        df[col] = pd.to_numeric(df[col], errors="coerce")

Stage 6: Handle Missing Values

Missing values in numeric columns were replaced with 0 to prevent calculation errors and maintain realistic totals.

In [ ]:
# -----------------------------------
# HANDLE MISSING VALUES
# -----------------------------------
df.fillna(0, inplace=True)

Stage 7: Save Clean Dataset

Saved the cleaned dataset as military_cleaned_ready.csv, ready for KPI analysis and dashboard creation.

In [ ]:
# -----------------------------------
# SAVE CLEANED DATA
# -----------------------------------
df.to_csv("military_cleaned_ready.csv", index=False)

print("✅ Stock & readiness separated successfully")

What was done :

The dataset is now fully cleaned, with stock and readiness values separated, financial fields converted to numeric format, formatting inconsistencies removed, and missing values handled — making it ready for accurate military strength and KPI analysis.